In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import sys
from PIL import Image

In [ ]:
# Add backend to path so we can use existing DiagramDetector and Feature Engineering
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', 'backend')))
from services.diagram_detector import DiagramDetector
import feature_engineering as fe
from pdf2image import convert_from_path

In [ ]:
def process_excel_files(base_costing_dir):
    """
    Parses the Quotation+BOM excel file and individual RFQ subdirectories
    to extract actual target costs and match them with their respective PDFs.
    """
    excel_path = os.path.join(base_costing_dir, "Quotation+BOM all of package PC4100.xlsx")
    rfq_dir = os.path.join(base_costing_dir, "Phase 2 RFQ package PC4100")
    
    if not os.path.exists(excel_path):
        print(f"Error: Could not find {excel_path}")
        return pd.DataFrame()
        
    print(f"Loading master excel file: {excel_path}")
    xls = pd.ExcelFile(excel_path)
    sheet_names = xls.sheet_names
    
    # We want to extract costs from sheets like "Item-1_AY30013"
    item_sheets = [s for s in sheet_names if s.startswith("Item-")]
    
    detector = DiagramDetector()
    data_rows = []
    
    for sheet in item_sheets:
        df_sheet = pd.read_excel(xls, sheet_name=sheet)
        
        # Example parsing logic: Since excel structure can vary, we look for keywords in columns
        # In a real heavy-duty script we'd pinpoint exactly the cell, but here we do a keyword search
        # for Total Cost, Material Cost, etc.
        
        # Heuristic search for totals
        total_cost = 0.0
        material_cost = 0.0
        machining_cost = 0.0
        
        # Iterate over cells to find cost totals
        for row in range(min(50, len(df_sheet))):
            for col in df_sheet.columns:
                cell_val = str(df_sheet.loc[row, col]).lower()
                if 'total' in cell_val and 'cost' in cell_val:
                    try:
                        # try to get the next column value
                        val = float(df_sheet.iloc[row, df_sheet.columns.get_loc(col) + 1])
                        total_cost = max(total_cost, val)
                    except:
                        pass
                if 'material' in cell_val and 'cost' in cell_val:
                    try:
                        val = float(df_sheet.iloc[row, df_sheet.columns.get_loc(col) + 1])
                        material_cost = max(material_cost, val)
                    except:
                        pass
                if 'machining' in cell_val and 'cost' in cell_val:
                    try:
                        val = float(df_sheet.iloc[row, df_sheet.columns.get_loc(col) + 1])
                        machining_cost = max(machining_cost, val)
                    except:
                        pass
        
        # Extract the part ID (e.g. AY30013)
        part_id = sheet.split("_")[1] if "_" in sheet else sheet
        
        # Find matching PDF in Phase 2 RFQ directory
        # The folders usually contain the part ID
        matching_dirs = [d for d in os.listdir(rfq_dir) if part_id.lower() in d.lower() and os.path.isdir(os.path.join(rfq_dir, d))]
        
        pdfs = []
        if matching_dirs:
            target_dir = os.path.join(rfq_dir, matching_dirs[0])
            pdfs = glob.glob(os.path.join(target_dir, "*.pdf"))
            
        print(f"Sheet {sheet} -> Part {part_id} -> Found {len(pdfs)} PDFs. Total Cost: {total_cost}")
        
        for pdf_path in pdfs:
            print(f"  Processing {pdf_path}...")
            # Extract image from PDF
            images = convert_from_path(pdf_path, first_page=1, last_page=1)
            if not images:
                continue
                
            img = images[0]
            analysis = detector.analyze_image(img)
            
            # Combine into a row that matches feature_names
            # We must set defaults for dimensions and materials since OCR might fail or we didn't hook it up fully yet
            
            # We set some proxy values for now for text-based features
            material_code = 1 # Assume steel
            density = 7.85
            machinability = 0.6
            cost_per_kg = 0.8
            
            # Dimensions placeholder (in a full setup, use Tesseract OCR to extract dimensions)
            # For now, base it on the complexity and area
            length = 500.0
            width = 300.0
            height = 100.0
            volume = length * width * height
            surface_area = 2 * (length * width + width * height + length * height)
            aspect_ratio = length / width
            
            row = {
                "length": length, "width": width, "height": height,
                "diameter": 0.0, "thickness": 10.0,
                "volume": volume, "surface_area": surface_area, "aspect_ratio": aspect_ratio,
                "material_code": material_code, "density": density,
                "machinability_index": machinability, "material_cost_per_kg": cost_per_kg,
                "num_operations": 3, "tolerance_severity": 20.0,
                "surface_finish_code": 2, "coating_code": 1,
                "hole_count": analysis.hole_count,
                "slot_count": analysis.slot_count,
                "pocket_count": analysis.pocket_count,
                "fillet_count": analysis.fillet_count,
                "chamfer_count": analysis.chamfer_count,
                "complexity_score": analysis.complexity_score,
                "contour_count": analysis.contour_count,
                "symmetry_score": analysis.symmetry_score,
                "num_drawing_views": analysis.num_drawing_views,
                
                # Targets
                "total_cost": total_cost if total_cost > 0 else 15000.0, # Default if parse failed
                "raw_material_cost": material_cost if material_cost > 0 else total_cost * 0.4,
                "machining_cost": machining_cost if machining_cost > 0 else total_cost * 0.3,
                "manpower_cost": total_cost * 0.1,
                "coating_cost": total_cost * 0.05,
                "overhead_cost": total_cost * 0.1,
                "logistics_cost": total_cost * 0.05,
                "profit_margin": total_cost * 0.2
            }
            data_rows.append(row)

    if data_rows:
        df = pd.DataFrame(data_rows)
        out_path = os.path.join(os.path.dirname(__file__), "training_data", "real_training_data.csv")
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        df.to_csv(out_path, index=False)
        print(f"Extracted {len(df)} real data samples to {out_path}")
        return df
    else:
        print("No data extracted.")
        return pd.DataFrame()

In [ ]:
if __name__ == "__main__":
    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "Costing"))
    process_excel_files(base_dir)